### RAG Pipeline - data ingestion to vector

In [1]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

c:\Users\USER\Downloads\RAG_CRASH_COURSE\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)

    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents=loader.load()

            for doc in documents:
                doc.metadata['source_file']=pdf_file.name
                doc.metadata['file_type']='pdf'

            all_documents.extend(documents)
            print(f" ✅ Loaded {len(documents)} pages")
        
        except Exception as e:
            print(f" ❌ Error: {e}")

    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all pdfs in the data dictionary
all_pdf_documents = process_all_pdfs("../data")


Found 3 PDF files to process

Processing: GuitarZero2Hero_EBook.pdf
 ✅ Loaded 43 pages

Processing: Longwood Seminar Music Reading Pack.pdf
 ✅ Loaded 46 pages

Processing: music-theory.pdf
 ✅ Loaded 222 pages

Total documents loaded: 311


In [3]:
### Text splitting into chunks

def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap,
        length_function = len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    # show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")

    return split_docs

In [4]:
chunks = split_documents(all_pdf_documents)
chunks

Split 311 documents into 527 chunks

Example chunk:
Content: Chord & Songwriting 
Cheat Sheet
The ultimate quick start 
guitar resource for beginners....
Metadata: {'producer': 'macOS Version 12.5.1 (Build 21G83) Quartz PDFContext, AppendMode 1.1', 'creator': 'Adobe InDesign 18.1 (Macintosh)', 'creationdate': "D:20230804042503Z00'00'", 'trapped': '/False', 'moddate': "D:20230804042640Z00'00'", 'source': '..\\data\\pdf\\GuitarZero2Hero_EBook.pdf', 'total_pages': 43, 'page': 0, 'page_label': '1', 'source_file': 'GuitarZero2Hero_EBook.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'macOS Version 12.5.1 (Build 21G83) Quartz PDFContext, AppendMode 1.1', 'creator': 'Adobe InDesign 18.1 (Macintosh)', 'creationdate': "D:20230804042503Z00'00'", 'trapped': '/False', 'moddate': "D:20230804042640Z00'00'", 'source': '..\\data\\pdf\\GuitarZero2Hero_EBook.pdf', 'total_pages': 43, 'page': 0, 'page_label': '1', 'source_file': 'GuitarZero2Hero_EBook.pdf', 'file_type': 'pdf'}, page_content='Chord & Songwriting \nCheat Sheet\nThe ultimate quick start \nguitar resource for beginners.'),
 Document(metadata={'producer': 'macOS Version 12.5.1 (Build 21G83) Quartz PDFContext, AppendMode 1.1', 'creator': 'Adobe InDesign 18.1 (Macintosh)', 'creationdate': "D:20230804042503Z00'00'", 'trapped': '/False', 'moddate': "D:20230804042640Z00'00'", 'source': '..\\data\\pdf\\GuitarZero2Hero_EBook.pdf', 'total_pages': 43, 'page': 1, 'page_label': '2', 'source_file': 'GuitarZero2Hero_EBook.pdf', 'file_type': 'pdf'}, page_content='Chord & Songwriting Cheat Sheet\n© 2

### Embedding & VectorStoreDB

In [5]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [6]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""

    def __init__(self, model_name: str="all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name = HuggingFace model name for sentence embeddings
        """
        self.model_name=model_name
        self.model=None
        self._load_model()
    
    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading Embedding Model: {self.model_name}")
            self.model=SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding Dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}:{e}")
            raise

    def generate_embeddings(self, texts: List[str])->np.ndarray:
        """
        Generate embeddings for a list of texts

        Args:
            texts: List of text strings to embed

        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")

        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings=self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings
    
    # def get_embedding_dimension(self)->int:
    #     """Get the embedding dimension of the model"""
    #     if not self.model:
    #         raise ValueError("Model not loaded")
    #     return self.model.get_sentence_embedding_dimension()

embedding_manager = EmbeddingManager()
embedding_manager


Loading Embedding Model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3817.28it/s]


Model loaded successfully. Embedding Dimension: 384


C:\Users\USER\AppData\Local\Temp\ipykernel_6784\3017927795.py:20: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding Dimension: {self.model.get_sentence_embedding_dimension()}")


### Vector store

In [7]:
class VectorStore:
    """
    Manages document embeddings in a ChromaDB vector store
    
    Chroma DB is an open-source vector database used to store & search embeddings
    """
    
    def __init__(self, collection_name: str="pdf_documents", persist_directory: str="../data/vector_store"):
        """
        Initialize the vector store

        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        
        """
        self.collection_name = collection_name
        # whatever vector store is going to create will be saved in hard disk
        self.persist_directory = persist_directory
        self.client = None
        self.collection=None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create peristent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client=chromadb.PersistentClient(path=self.persist_directory)

            # Get or create collection
            self.collection=self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
        
        except Exception as e:
            printf("Error initializing vector store: {e}")
            raise
        
    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store

        Args:
            documents: List of Langchain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents)!=len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")

        print(f"Adding {len(documents)} to vector store...")

        # Prepare data for ChromaDB
        ids=[]
        metadatas=[]
        documents_text=[]
        embeddings_list=[]

        for i, (doc,embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)

            # Document Content
            documents_text.append(doc.page_content)

            # Embedding
            embeddings_list.append(embedding.tolist())

        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents = documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
        
        except Exception as e:
            print(f"Error adding docs to vector store: {e}")
            raise

vectorstore = VectorStore()
vectorstore


Vector store initialized. Collection: pdf_documents
Existing documents in collection: 585


### Text -> Embeddings -> VectorDB

In [8]:
texts=[doc.page_content for doc in chunks]

# Generate the Embeddings
embeddings=embedding_manager.generate_embeddings(texts)

# Store in the vector db
vectorstore.add_documents(chunks,embeddings)

Generating embeddings for 527 texts...


Batches: 100%|██████████| 17/17 [00:15<00:00,  1.11it/s]


Generated embeddings with shape: (527, 384)
Adding 527 to vector store...
Successfully added 527 documents to vector store
Total documents in collection: 1112


### Retriever Pipeline 

In [9]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""

    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever

        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager
    
    def retrieve(self, query:str, top_k: int=5, score_threshold: float=0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query

        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
        
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """

        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")

        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        # Search in the vector store
        try: 
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )

            # Process results
            retrieved_docs=[]

            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas=results['metadatas'][0]
                distances = results['distances'][0]
                ids=results['ids'][0]

                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1-distance

                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i+1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")

            return retrieved_docs

        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []


rag_retriever = RAGRetriever(vectorstore, embedding_manager)

In [10]:
## Call rag retrieve with a query
rag_retriever.retrieve("How many chords are there in music?")

Retrieving documents for query: 'How many chords are there in music?'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 85.00it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_009aaa8c_17',
  'content': 'CHAPTER 2 — CHORD CHARTS\n12\nThe Beginner Guitar Hero’s \nEssential Chord Chart\nChapter 2 — Chord Charts\nThe chord chart on the following page provides you \nwith the 12 most essential chords shapes that EVERY \nguitarist should know, along with real life photo’s of \nhow they should look.\nThese are the chords that over my 14 years of playing \nand teaching experience appear in more songs than \nany other chords.\nIf you’re a beginner, study this, print it out, stick it on \nyour bathroom door, stick it beside your bed, stick it \non your fridge door, hell stick it anywhere you’ll be able \nto see it daily!\nThere will be 2 versions of these charts\n• The first with photos\n• The second without photos\nOnce you’ve learnt the 16 chords on the next page \nyou’ve basically learnt the guitar chord shapes used in \nroughly 60-80% of any song you’ve ever heard!\nWant proof? If you’ve watched my YouTube video \ntutorials, you’ll notice that 90-95% 

### VectorDB Context pipeline with LLM output 

In [13]:
### Simple RAG pipeline with Groq LLM
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

### Initialize the groq LLM (set your GROQ_API_KEY in environment)
groq_api_key = os.getenv("GROQ_API_KEY")

llm = ChatGroq(groq_api_key=groq_api_key, model_name="llama-3.3-70b-versatile", temperature=0.1, max_tokens=1024)

def rag_simple(query, retriever, llm, top_k=3):
    ## retrieve the context
    results = retriever.retrieve(query, top_k=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question"
    
    ## generate the answer using GROQ LLM
    prompt=f"""Use the following context to answer the question concisely.
        Context:{context}
        Question:{query}
        Answer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])
    return response.content

In [14]:
answer = rag_simple("How many chords are there in music?", rag_retriever, llm)
print(answer)
        

Retrieving documents for query: 'How many chords are there in music?'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 82.11it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


The text doesn't provide a specific number of chords in music, but it mentions that learning 16 essential chords can cover roughly 60-80% of any song.


### Enhanced RAG Pipeline Features

In [16]:
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - answer, sources, confidence_score
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview':doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])

    # Generate answer
    prompt = f"""Use the following context to answer the question concisely. \nContext:\n{context}\n\nQuestion:{query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])

    output = {
        'answer':response.content,
        'sources':sources,
        'confidence':confidence
    }

    if return_context:
        output['context']=context
    return output

# Example usage:
results = rag_advanced("How many chords are there in music?", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer", results['answer'])
print("Sources:", results['sources'] )
print("Confidence:", results['confidence'])
print("Context Preview:", results['context'][:300])


Retrieving documents for query: 'How many chords are there in music?'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 131.68it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


Answer The context doesn't provide a specific number of chords in music, but it mentions that learning 16 essential chords can cover roughly 60-80% of any song.
Sources: [{'source': 'GuitarZero2Hero_EBook.pdf', 'page': 11, 'score': 0.25394344329833984, 'preview': 'CHAPTER 2 — CHORD CHARTS\n12\nThe Beginner Guitar Hero’s \nEssential Chord Chart\nChapter 2 — Chord Charts\nThe chord chart on the following page provides you \nwith the 12 most essential chords shapes that EVERY \nguitarist should know, along with real life photo’s of \nhow they should look.\nThese are the ...'}, {'source': 'GuitarZero2Hero_EBook.pdf', 'page': 11, 'score': 0.25394344329833984, 'preview': 'CHAPTER 2 — CHORD CHARTS\n12\nThe Beginner Guitar Hero’s \nEssential Chord Chart\nChapter 2 — Chord Charts\nThe chord chart on the following page provides you \nwith the 12 most essential chords shapes that EVERY \nguitarist should know, along with real life photo’s of \nhow they should look.\nThese are the ...'}, {'source'